# 03 · Modelo SVM

Objetivo: entrenar y ajustar un clasificador Support Vector Machine.

- Búsqueda de hiperparámetros (C, kernel, gamma)
- Validación cruzada
- Métricas de desempeño
- Persistencia del modelo entrenado

In [1]:

import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    classification_report
)

# Rutas del proyecto
carpeta_datos = Path("../data/procesado")
carpeta_modelos = Path("../models")
carpeta_modelos.mkdir(parents=True, exist_ok=True)

# Cargar datos de entrenamiento y prueba
X_train = np.load(carpeta_datos / "X_train.npy")
X_test = np.load(carpeta_datos / "X_test.npy")
y_train = np.load(carpeta_datos / "y_train.npy")
y_test = np.load(carpeta_datos / "y_test.npy")

# Etiquetas que vamos a predecir
columnas_sabores = [
    "dulce", "salado", "amargo", "acido", "umami"
]

print("Datos cargados correctamente")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


Datos cargados correctamente
X_train: (2379, 1814)
X_test: (1020, 1814)
y_train: (2379, 5)
y_test: (1020, 5)


In [2]:

# Crear el modelo SVM multietiqueta
modelo_svm = Pipeline([
    ("escalador", StandardScaler()),
    ("clasificador", OneVsRestClassifier(
        SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced"
        ),
        n_jobs=-1
    ))
])

# Entrenar el modelo
print("Entrenando SVM...")
modelo_svm.fit(X_train, y_train)

# Guardar el modelo
joblib.dump(
    modelo_svm,
    carpeta_modelos / "svm_multietiqueta.joblib"
)

print("Modelo SVM entrenado y guardado.")


Entrenando SVM...
Modelo SVM entrenado y guardado.


In [3]:

# Predecir los sabores del conjunto de prueba
y_pred_svm = modelo_svm.predict(X_test)

# Calcular métricas generales
metricas_svm = {
    "F1 macro": f1_score(
        y_test, y_pred_svm, average="macro", zero_division=0
    ),
    "F1 micro": f1_score(
        y_test, y_pred_svm, average="micro", zero_division=0
    ),
    "Precision macro": precision_score(
        y_test, y_pred_svm, average="macro", zero_division=0
    ),
    "Recall macro": recall_score(
        y_test, y_pred_svm, average="macro", zero_division=0
    ),
    "Hamming loss": hamming_loss(y_test, y_pred_svm)
}

print("MÉTRICAS DEL MODELO SVM")
for nombre, valor in metricas_svm.items():
    print(f"{nombre}: {valor:.4f}")

print("\nREPORTE POR SABOR")
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=columnas_sabores,
    zero_division=0
))


MÉTRICAS DEL MODELO SVM
F1 macro: 0.5110
F1 micro: 0.7196
Precision macro: 0.5022
Recall macro: 0.5209
Hamming loss: 0.2598

REPORTE POR SABOR
              precision    recall  f1-score   support

       dulce       0.44      0.42      0.43       344
      salado       0.86      0.93      0.89       853
      amargo       0.00      0.00      0.00        61
       acido       0.41      0.45      0.43       329
       umami       0.80      0.81      0.81       761

   micro avg       0.72      0.72      0.72      2348
   macro avg       0.50      0.52      0.51      2348
weighted avg       0.69      0.72      0.71      2348
 samples avg       0.73      0.73      0.70      2348



In [4]:

# Guardar las predicciones
np.save(
    carpeta_datos / "y_pred_svm.npy",
    y_pred_svm
)

# Guardar las métricas
pd.DataFrame([
    {"modelo": "SVM", "metrica": nombre, "valor": valor}
    for nombre, valor in metricas_svm.items()
]).to_csv(
    carpeta_datos / "metricas_svm.csv",
    index=False
)

print("Predicciones y métricas guardadas correctamente.")


Predicciones y métricas guardadas correctamente.
